# Prompt Compression Testing

## Aim

Make LLM calls cheaper by reducing the number of input tokens being sent to an LLM without affecting quality or increasing output tokens.

## Hypothesis

We can reduce the number of input tokens being sent to an LLM whist maintaining output quality


## Experiment Setup (Questions not Set in Stone)

- Do we have non-compressed prompt with response as a reference
    - Assume this prompt plus reponse is our control
    - Then compare compression methods against our control
        - Look at input and output length (in tokens)
        - Look at output quality (how do we do this)

## How do we assess the quality of an LLM output?

### 1. BERTScore (How much the "meaning" drifted)
- **What it measures**: 
    - Uses a pre-trained transformer (like RoBERTa) to convert both Control Output and Compressed Output into high-dimensional vectors
    - Calculates the Cosine Similarity between these vectors
    - Score ranges from 0 to 1
- **Why use it**: 
    - Instead of asking an LLM "Is this good?", it's a purely mathematical comparison of meaning
    - If compressed prompt caused the model to miss a key instruction, the vector will "drift," and the score will drop predictably
    - The "Regression" feel: You can set a strict threshold (e.g., 0.92) and anything below that is a failure
- **Note**: Could we do this to the input and the output too?
- **Exploration note**: MoverScore is suggested as an improvement on this framework

### 2. ROUGE Score
- **What it measures**: 
    - Primarily used for comparing a computer-generated summary of text with a reference (i.e. human-generated) summary
    - Mostly used for evaluating text-summarization tasks
    - Counts exact word overlaps
- **Limitation**: ROUGE is often "dumb"—it only counts exact word overlaps
- **Exploration note**: METEOR is suggested as an improved metric

### 3. KL Divergence
- **What it measures**: Statistical divergence between probability distributions of outputs
- **Requirements**: Can only be used if we can get the log probabilities for the outputs
- **Current limitation**: 
    - Anthropic don't support this
    - Gemini suggests GPT4 models (non-reasoning do) support it
    - **Suggest we pause this one for now** as we can't benchmark Anthropic models which Defra want to use

### 4. Variation Rate (VR) — Specific to Compression
- **What it measures**: The percentage of words in the "Compressed" output that do not appear in the "Control" output
- **Why use it**: 
    - If you are using LLMLingua, this is your most important "Safety" metric
    - Since LLMLingua is meant to be extractive (removing tokens, not rewriting them), the compressed prompt should not cause the model to "invent" new vocabulary that wasn't in the original
    - A high VR suggests compression was too aggressive, causing the model to hallucinate or "fill in the blanks" rather than following the original context

### 5. METEOR (Better than ROUGE)
- **What it measures**: Scores output based on:
    - Exact matches
    - Stemming (e.g., "running" and "runs" match)
    - Synonyms (using WordNet)
- **Why use it**: 
    - METEOR was designed to fix ROUGE's limitations
    - Has significantly higher correlation with human judgment than ROUGE or BLEU
    - Understands that two different words can mean the same thing

### 6. MoverScore (The "Upgrade" to BERTScore)
- **What it measures**: 
    - Uses "Word Mover's Distance" on contextual embeddings
    - Asks: "How much effort would it take to transform the Control output into the Compressed output?"
- **Why use it**: 
    - While BERTScore looks at how similar individual tokens are, MoverScore looks at the whole "flow" of the sentence
    - More robust against sentence reordering
    - If compressed prompt causes the LLM to provide same information in different order, BERTScore might penalize it slightly, but MoverScore correctly sees that meaning is intact
- Practicality - the MoverScore library hasn't been updated and the BERTScore library is actively maintained. Meaning we should probably use BERTScore instead


### 7. Question-Answer Generation (QAG) Score
- **The ultimate "Information Fidelity" test** - scores the knowledge, not the text
- **The Step**: Take Control Output and generate 3-5 "fact-check" questions based on it
- **The Test**: Ask those same questions to the Compressed Output
- **The Metric**: % of questions answered correctly
- **Why it works**: 
    - Binary, deterministic result (Pass/Fail)
    - Proves compressed prompt didn't delete "key" information needed to solve the task

### 8. Pass@k
- **What it is**: A metric used to evaluate the probability that at least one of the top $k$ generated responses is correct
- **How it works**: 
    - In a traditional setting, you might generate 100 samples ($n=100$) for a single prompt and see how many pass a unit test ($c$)
    - The formula calculates the probability that if you only picked $k$ samples, at least one would be correct
    - For your experiment, you will likely use **Pass@1** - the most "regression-like" version: you run the compressed prompt once and see if it succeeds
- **Why it's vital for your LLMLingua experiment**:
    - Prompt compression often introduces "stochastic noise"
    - A compressed prompt might work 80% of the time but fail 20% of the time because a critical "hint" or "constraint" was removed
    - **The Control**: Should have a high Pass@1 (e.g., 95%)
    - **The Compressed**: If the Pass@1 drops to 60%, your compression technique is "breaking" the model's reasoning chain, even if the BERTScore remains high

### 9. Constraint Satisfaction Rate (CSR)
- **What it measures**: If your prompt has specific formatting requirements (e.g., "Output in JSON," "Use no more than 3 sentences," "Include the reference ID"), CSR measures the percentage of those constraints that were met
- **How to automate**: Use a simple Python script with Regex or a JSON validator
- **Why it matters**: 
    - Compression often trims the "instructional" tokens that tell the model how to format the answer
    - If your JSON fails to parse after compression, the quality has dropped to zero for that task
- **Use case**: Provides binary, "hard" data points for capability assessment

### 10. Answer Consistency (Self-Consistency)
- **What it is**: A proxy for "Model Confidence" when you don't have logprobs (like with Anthropic)
- **The Test**: Run the Compressed Prompt 5 times at a temperature of 0.7
- **The Metric**: Do all 5 outputs say the same thing?
- **The Logic**: 
    - If the 5 outputs are wildly different, the prompt is too "vague" due to compression
    - If they are identical, the prompt is still "dense" enough to guide the model to a singular logical conclusion
- **Use case**: Provides binary, "hard" data points for capability assessment

---

## Summary Comparison Table

| Metric | Type | Best For... | Benefit |
|--------|------|-------------|---------|
| **BERTScore** | Semantic | Meaning Preservation | Sets strict thresholds for semantic drift (e.g., >0.92 = pass). |
| **ROUGE** | Lexical | Exact Word Matching | Simple baseline for summarization; easy to implement. |
| **KL Divergence** | Probabilistic | Distribution Shift | Quantifies how much output probability changed (requires logprobs). |
| **Variation Rate (VR)** | Lexical | Hallucination Detection | Catches if compression causes model to invent new vocabulary. |
| **METEOR** | Lexical+ | Nuanced Comparison | Understands synonyms, stemming, and word forms beyond exact matches. |
| **MoverScore** | Semantic | Structural Drift | Better at handling reordered sentences than BERTScore. |
| **QAG Score** | Functional | Knowledge Retention | Proves key facts weren't lost; binary pass/fail for information fidelity. |
| **Pass@k** | Functional | Reliability Testing | Reveals if compression breaks reasoning chains despite good similarity scores. |
| **CSR** | Structural | Format Compliance | Ensures structured outputs (JSON, length limits) still work after compression. |
| **Answer Consistency** | Functional | Prompt Specificity | Detects if compression made prompt too vague or ambiguous. |


## LLM Lingua 

This methodology is extractive meaning that it determines which words are not required for the prompt. It does not rewrite the prompt. Therefore a limitation is that it can remove valuble supporting words. A rewriting approach might be useful to explore.

In [ ]:
### Meteor Score

from meteor_scorer import (
    setup_nltk_resources,
    calculate_meteor
)


# Control output (from non-compressed prompt)
control_output = "The API returns a JSON object containing user data, including name, email, and registration date"

# Compressed output (from compressed prompt)
compressed_output = "API provides JSON with user information: name, email, signup date"

result = calculate_meteor(control_output, compressed_output)

print("Control Output:")
print(f"  {control_output}")
print(f"\nCompressed Output:")
print(f"  {compressed_output}")
print(f"\nResults:")
print(f"  METEOR Score: {result['score']:.4f}")
print(f"  Control tokens: {result['reference_length']}")
print(f"  Compressed tokens: {result['hypothesis_length']}")
print(f"  Length ratio: {result['length_ratio']:.2%}")
print(f"  Token reduction: {(result['token_reduction']):.2%}")

Control Output:
  The API returns a JSON object containing user data, including name, email, and registration date

Compressed Output:
  API provides JSON with user information: name, email, signup date

Results:
  METEOR Score: 0.4013
  Control tokens: 18
  Compressed tokens: 13
  Length ratio: 72.22%
  Token reduction: 27.78%


{'score': 0.40133928571428573,
 'reference_length': 18,
 'hypothesis_length': 13,
 'length_ratio': 0.7222222222222222,
 'token_reduction': 0.2777777777777778}

In [5]:
### BERTScore

from bertscore_scorer import calculate_bertscore

# Control output (from non-compressed prompt)
control_output = "The API returns a JSON object containing user data, including name, email, and registration date"
# Compressed output (from compressed prompt)
compressed_output = "API provides JSON with user information: name, email, signup date"

result = calculate_bertscore(control_output, compressed_output)

print("Control Output:")
print(f"  {control_output}")
print(f"\nCompressed Output:")
print(f"  {compressed_output}")
print(f"\nResults:")
print(f"  BERTScore F1: {result['f1']:.4f}")
print(f"  Precision: {result['precision']:.4f}")
print(f"  Recall: {result['recall']:.4f}")
print(f"  Control tokens: {result['reference_length']}")
print(f"  Compressed tokens: {result['hypothesis_length']}")
print(f"  Length ratio: {result['length_ratio']:.2%}")
print(f"  Token reduction: {result['token_reduction']:.2%}")

/Users/adamfletcher/Documents/GitHub/LLM-prompt_compression/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Control Output:
  The API returns a JSON object containing user data, including name, email, and registration date

Compressed Output:
  API provides JSON with user information: name, email, signup date

Results:
  BERTScore F1: 0.8931
  Precision: 0.9036
  Recall: 0.8828
  Control tokens: 15
  Compressed tokens: 10
  Length ratio: 66.67%
  Token reduction: 33.33%
